# LLM 推理：CoT、自洽投票与涌现度量


> 前几讲各处理 Agent 的一个环节：11 讲管记忆，10 讲把任务落成写代码，14 讲做评测。它们都默认一个前提——模型单次续写就能给出合理的下一步。可一旦任务本身要多步推理才算对，这个前提就站不住，扩大模型规模也看不到明显改善。
>
> 本讲补的正是这个缺口：推理能力从哪里来，以及不动权重的前提下怎么把它调出来。我们先亲手实现"让模型把中间步骤写出来"的提示，再做消融对比，然后实现"多试几次再投票"的自洽解码，最后回到一个争议——某些能力随模型变大突然出现，这件事可能只是度量方式造成的假象。

先看一道小学题：罗杰有 5 个网球，又买来 2 罐、每罐 3 个，一共几个？

模型在标准提示下直接报一个数字，常常猜错——它要一步把"先乘再加"和最终结果都算对，错一点全盘皆错。

换成让它把过程写出来：先算新买的球，2×3=6；再把原来的 5 个加上去，5+6=11；最后才写最终答案。每一步都摆在上下文里，模型算下一步时随时能读到上一步的结果。

同样是这道题，只是把"直接报答案"换成"先写步骤再报答案"，答案常常就从错变对。这种把推理过程一步步写出来的提示格式，就是**思维链**（chain-of-thought）。

CoT 论文（Wei 等人，2022）在 GSM8K 这个小学数学题集上发现一个矛盾：同一个模型、同样的参数，只是把提示里的样例从"题目—答案"改成"题目—推理步骤—答案"，准确率就明显上涨。这说明一部分推理能力本来就藏在权重里，只是缺少一种合适的提示格式把它调出来。

这一讲要做的第一件事，就是亲手验证这个矛盾：构造两组提示，一组直接要答案，一组让模型先写步骤，然后对比两组输出。第一节从构造这两组提示开始。

这一节要做的是一件很具体的事：准备一个小题库，写出两组提示，一组直接要答案，一组让模型先写中间步骤，然后看模型各怎么回答。前面我们说过，分步思考能把模型已有的能力调出来，这一节就动手验证这句话。原理上一句话就能说清：预训练把语言规律写进了权重，推理能力也包含其中，提示格式决定模型以什么方式调用这些权重。真实系统里，我们发给 ChatGPT 的那段话就是提示，本节先看清它的结构。

**实验语料**：题库含两道演示题，答案都是整数。两套样例各写两条，差别只在是否包含中间推理步骤。

标准格式（⟨问题, 答案⟩）：

```text
问：罗杰有 5 个网球，他又买来 2 罐，每罐 3 个。现在有多少个？
答：最终答案是 11。
```

思维链格式（⟨问题, 推理步骤, 答案⟩）：

```text
问：罗杰有 5 个网球，他又买来 2 罐，每罐 3 个。现在有多少个？
答：他先算出新买的球：2 罐乘以每罐 3 个，得 6 个。再加上原来的 5 个，5 + 6 = 11。最终答案是 11。
```


**提示模板：把样例和问题拼起来**

两套样例只差一个环节。把结构抽出来，就是两个模板：

```text
标准格式：  问：<题目>
           答：<最终答案>

思维链格式：问：<题目>
           答：<推理步骤>
           最终答案是 <最终答案>。
```

两套提示都以"问："开头、以"答："结尾。模型要做的是接在最后一个"答："后面继续写：标准提示下它只写一个数字，思维链提示下它先写中间步骤，再写最终答案。

把几条样例和待答问题拼成一段话，让模型接着写，这种用法叫 few-shot：先给几个例子，再问一个问题。一条例子太少，模型分不清哪些是题目、哪些是步骤；两条例子就把"先算后答"的模式固定下来。样例不参与训练，它们只是提示字符串的一部分，模型照抄的是结构，不是内容。

这样设计有三个原因。第一，答案放在最后，模型生成最终数字时，前面写的每一步都留在上下文里，可以随时回头引用。中间结果写进了文本，不必记在模型内部的向量里，这降低了"记不住上一步"的风险。

第二，每个中间步骤都是一句通顺的自然语言。模型写完上一步再写下一步，倾向延续自己刚写的内容，偏离到无关话题的可能性更小。

第三，"最终答案是"是一个稳定的收尾标记。后面的解析函数靠它把答案从长篇输出里精确地提取出来，多数投票也依赖同一个标记。

In [ ]:
import numpy as np

np.random.seed(42)  # 保证实验可复现

# 两道用于演示的题目，真实答案分别为 11 与 19
QUESTIONS = [
    "罗杰有 5 个网球，他又买来 2 罐，每罐 3 个。现在有多少个？",
    "妈妈买了 4 袋苹果，每袋 6 个，吃掉 5 个后还剩多少个？",
]
TRUTH = [11, 19]

# 标准提示样例：只给题目与答案
STANDARD_EXEMPLARS = [
    "问：小明有 3 支笔，又买了 4 支。一共有几支？\n答：最终答案是 7。",
    "问：书架上原有 9 本书，拿走 3 本。还剩几本？\n答：最终答案是 6。",
]

# 思维链提示样例：中间推理步骤放在答案之前
COT_EXEMPLARS = [
    "问：小明有 3 支笔，又买了 4 支。一共有几支？\n"
    "答：他原来有 3 支，买来 4 支后是 3 + 4 = 7 支。最终答案是 7。",
    "问：书架上原有 9 本书，拿走 3 本。还剩几本？\n"
    "答：拿走代表减去，9 - 3 = 6。最终答案是 6。",
]


def build_prompt(question, exemplars, tail=""):
    """把样例与待回答问题拼成一个提示，tail 可放引导语。

    exemplars: 样例字符串列表；question: 待回答的问题。
    提示以"答："结尾，让模型接着输出。
    """
    parts = exemplars + ["问：" + question + "\n" + tail + "答："]
    return "\n\n".join(parts)


print("标准提示示例：")
print(build_prompt(QUESTIONS[0], STANDARD_EXEMPLARS))
print("\n思维链提示示例：")
print(build_prompt(QUESTIONS[0], COT_EXEMPLARS))


两条提示的最大差别在输出路径。标准提示下，模型一步输出最终数字；思维链提示下，模型先输出逐步计算，再用"最终答案是"收尾。下面实现一个解析函数，从这两类输出里提取答案数字，然后用 llm_client 各跑一遍。没有配置 API key 时自动进入真实 API 演示，模型输出是占位文本，跑通的是整条解析与对比的管线。

In [ ]:
import os
import re
import sys

_root = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(_root, "llm_client.py")):
    _root = os.path.dirname(_root)
    if _root == os.path.dirname(_root):
        break
if _root not in sys.path:
    sys.path.insert(0, _root)

from llm_client import get_llm

client = get_llm()


def parse_answer(text):
    """从模型输出中提取答案数字，解析不到返回 None。

    优先匹配"最终答案是 N"这种固定收尾，其次匹配"答案是 N"，
    最后退回文本里最后一个整数。
    """
    for pat in [r"最终答案是\s*(\d+)", r"答案是\s*(\d+)", r"答[:：]?\s*(\d+)"]:
        m = re.search(pat, text)
        if m:
            return int(m.group(1))
    nums = re.findall(r"\d+", text)
    return int(nums[-1]) if nums else None


def run_prompt(prompt):
    """调用模型返回原始输出文本，使用贪婪解码。"""
    return client.chat([{"role": "user", "content": prompt}], temperature=0.0)


for label, exemplars in [("标准提示", STANDARD_EXEMPLARS), ("思维链", COT_EXEMPLARS)]:
    print("=" * 22, label, "=" * 22)
    correct = 0
    for q, truth in zip(QUESTIONS, TRUTH):
        out = run_prompt(build_prompt(q, exemplars))
        ans = parse_answer(out)
        if ans == truth:
            correct += 1
        verdict = "命中" if ans == truth else ("未解析出数字" if ans is None else "未命中")
        print("题目：", q)
        print("输出：", out.replace("\n", " ")[:70])
        print(f"解析 {ans} / 真实 {truth} -> {verdict}\n")
    print(f"正确 {correct}/{len(QUESTIONS)}\n")

if False:
    print("真实 API 演示：模型输出为占位文本，上面的命中数不代表真实推理能力。")


真实 API 下，两组输出形成对比：思维链的回答里能看到中间步骤，标准回答只有结果。解析函数把两者都归约成最终答案数字，后续实验只关心这个数字。真实 API 演示下两条路径都是占位文本，不体现步骤差异，需要真实模型才能观察完整推理链。


上一节看到思维链有效，这一节要弄清这个增益来自哪一部分。一个直接的办法，是把中间步骤换一种形式，看效果还在不在。这种"去掉或替换一部分，观察变化"的实验方法叫消融（ablation）。CoT 论文做了一组消融：把中间步骤换成纯方程、换成省略号、或挪到答案之后，三种改法增益全部消失。剩下的唯一变量，是"用自然语言按顺序把步骤写出来"这件事本身。下面先用手算的玩具模型复现这个结论，再实现一种"多试几次再投票"的解码方法。

先建立数量感，也就是把"思维链有没有用"变成两个可以比的数字。设一道题需要 L 个顺序步骤。直接给答案的模型一次做对的概率是 p 直；分步模型每步做对概率是 p 步，最终正确需要 L 步全对，概率为 p 步的 L 次方。取 p 直 = 0.25、p 步 = 0.80、L = 3，先算两个数。

三种消融各自去掉中间步骤的一部分。只给方程：样例里只有最终算式，模型仍要一次算完；只给点：步骤用省略号占位，相当于只提示"可以多算"，不提示怎么算；推理在答案之后：先给最终答案再补推理，模型生成答案时用不到步骤。三种变体都回到"一次作答"，所以它们的概率都用 p 直。

In [ ]:
p_direct = 0.25
p_step = 0.80
L = 3
p_cot = p_step ** L

print(f"直接作答：{p_direct:.3f}")
print(f"分步作答：每步 {p_step:.2f}，{L} 步全对 = {p_step} ** {L} = {p_cot:.3f}")
print(f"两者差距：{p_cot - p_direct:.3f}")


In [ ]:
def accuracy_direct(n_trials, p_direct, seed):
    """直接作答：一次尝试，做对概率为 p_direct。"""
    rng = np.random.default_rng(seed)
    return float((rng.random(n_trials) < p_direct).mean())


def accuracy_steps(n_trials, p_step, L, seed):
    """分步作答：L 步全对才算对，每步做对概率为 p_step。"""
    rng = np.random.default_rng(seed)
    steps = rng.random((n_trials, L)) < p_step
    return float(steps.all(axis=1).mean())


import matplotlib.pyplot as plt

n_trials = 4000
acc = {
    "baseline": accuracy_direct(n_trials, p_direct, 1),
    "equation only": accuracy_direct(n_trials, p_direct, 2),
    "dots only": accuracy_direct(n_trials, p_direct, 3),
    "answer first": accuracy_direct(n_trials, p_direct, 4),
    "chain of thought": accuracy_steps(n_trials, p_step, L, 5),
}

fig, ax = plt.subplots(figsize=(6, 3.6))
ax.bar(list(acc), list(acc.values()), color=["#999999"] * 4 + ["#1f77b4"])
ax.set_ylabel("accuracy")
ax.set_title("Ablation of the intermediate steps")
ax.set_ylim(0, 1)
for i, (k, v) in enumerate(acc.items()):
    ax.text(i, v + 0.02, f"{v:.2f}", ha="center", fontsize=9)
plt.tight_layout()
plt.show()

print("关键观察：三种消融都贴着 baseline，只有 chain of thought 明显高出。")


**为什么把步骤写出来就有用**

消融实验确认了关键变量是"用自然语言按顺序写出步骤"。这一节解释这背后的三个机制。

第一个机制，是把中间结果从模型内部搬出来，写进文本。直接作答时，中间结果 2×3=6 只存在于模型内部，模型要先算出 6，再带着 6 去加 5，全靠内部状态一路接力，容易丢失。分步作答把 6 直接写进文本，模型生成下一步时随时能读到，不用再记。把中间结果放在外面、让模型随时能读，这种做法叫外部化工作记忆。

第二个机制是任务分解。一次做完时，每一步都可能出错，正确率是所有步骤的连乘。前面已经算过：一次做完是 0.25，分步每步 0.8、三步全对约 0.512。分步把"一次算对一个大计算"换成"三次各自算对一个小计算"，单步的难度和出错面都变小了。

第三个机制是连贯性。语言模型生成时倾向与已有上文一致。分步模型写完"5 + 6 = 11"之后，下一步几乎不会写出"最终答案是 3"，因为 3 与刚写下的 11 在叙述上不连贯。中间步骤成了约束后续输出的一个锚点。

需要澄清的是，思维链不能凭空造出能力。它调出的是权重里已经存在的推理，只是把调用方式从"一步"改成"多步"。这正是前面说的：一部分推理能力本来就存在，提示格式决定它能否被调用。

一次思维链解码只走一条推理路径，某一步出错就一路带到终点。改进的思路是让模型多走几条路：从同一个提示出发，多次采样得到好几条回答，每条回答都算出一个最终答案，最后数一数哪个答案出现得最多，最多的胜出。多条路径在同一个答案上彼此一致，这个做法叫自洽解码（self-consistency）。它为什么有效，直觉是这样：一道多步推理题，通常有好几条不同的正确路径都通向同一个答案，正确路径会在答案处汇合；而错误路径各有各的错法，很少同时指向同一个错误数字。我们先用一个只有两个答案的设定手算，再在多个错误答案的设定下模拟。

In [ ]:
from math import comb

p = 0.7
K = 3
# 两分类情形：正确票数严格过半才算多数
maj = sum(comb(K, i) * p ** i * (1 - p) ** (K - i)
          for i in range(K // 2 + 1, K + 1))
print(f"单次采样做对概率 {p:.2f}")
print(f"{K} 条路径两分类多数投票做对概率 {maj:.3f}")


**五条采样路径手算一遍**

多数投票的机制可以用五条路径直接算一遍。题目是"妈妈买了 4 袋苹果，每袋 6 个，吃掉 5 个后还剩多少个？"，真实答案是 19。同一个提示重复采样五次，模型五次给出不同的输出：

In [ ]:
def majority_accuracy(p, K, W, n_trials, seed):
    """模拟多数投票：每条路径以 p 做对，否则在 W 个错误答案中等概率取一。"""
    rng = np.random.default_rng(seed)
    hits = 0
    for _ in range(n_trials):
        correct = rng.random(K) < p
        wrong = rng.integers(1, W + 1, size=K)
        votes = np.where(correct, 0, wrong)
        counts = np.bincount(votes, minlength=W + 1)
        hits += counts.argmax() == 0
    return hits / n_trials


K_grid = range(1, 41)
fig, ax = plt.subplots(figsize=(6.4, 3.6))
for p in [0.5, 0.6, 0.7, 0.8]:
    acc = [majority_accuracy(p, K, W=8, n_trials=2000, seed=int(p * 100))
           for K in K_grid]
    ax.plot(list(K_grid), acc, label=f"p = {p:.1f}")
ax.axhline(1 / 8, color="gray", ls="--", lw=1)
ax.set_xlabel("number of sampled paths K")
ax.set_ylabel("majority vote accuracy")
ax.set_title("Majority vote over sampled paths")
ax.legend()
plt.tight_layout()
plt.show()

print("关键观察：只要单条路径做对概率高于 1/W，投票正确率随 K 快速逼近 1；")
print("错误路径被摊薄到多个错误类别，几乎不会在同一答案上扎堆。")


多数投票的前提是"同一种写法"算同一票。自由文本答案里，正确写法常有拼写出入，banana 被写成 bananna 就很常见。精确匹配会把这两种写法当成两个答案，票数被分散，可能输给一个碰巧扎堆的错误答案。

要修这个问题，先得有一个量来表示"两个写法有多像"。把一个字符串改成另一个，最少要做几次增、删、改，这个次数叫编辑距离。banana 和 bananna 只差一个字母，距离是 1；banana 和 grape 要全改，距离很大。改进的办法是：把编辑距离很近的写法合成一组，这一组叫一个簇，投票时按整簇的票数算。

下面从零实现编辑距离与簇投票。我们用的检验数据是合成的，也就是程序生成、用来模拟真实采样的答案，这种数据叫合成数据。下面在带拼写噪音的合成数据上，把簇投票和精确投票对比。

**不同写法把票数分散**

多数投票要求"同一个答案"计同一票。自由文本答案没有固定写法，正确的 banana 可能被写成 bananna 甚至 banan。下面手算五条路径，真实答案是 banana：

```text
路径 1：banana
路径 2：bananna
路径 3：banan
路径 4：grape
路径 5：grape
```

先按精确匹配数票：banana 1 票、bananna 1 票、banan 1 票、grape 2 票。grape 得票最多，一个错误答案胜出。问题在于正确的三种写法各自只有 1 票，票数被拼写差异分散了；grape 恰好出现两次，反而成了多数。

按编辑距离聚类后再投票：banana 与 bananna 距离 1，banana 与 banan 距离 1，banan 与 bananna 直接距离是 2，但通过 banana 连接，三个写法仍被并入同一簇。簇内票数合计 3 票，超过 grape 的 2 票。代表答案取簇内出现次数最多的写法，即 banana，正确。

聚类前后两次投票，结果从错变成对。多出来的信息只有一条规则：编辑距离不超过 1 的字符串被视为同一个答案。投票的对象由此从"字符串"升级为"同一个答案的不同写法"这个抽象概念。

In [ ]:
from collections import Counter


def edit_distance(a, b):
    """计算字符串 a 到 b 的编辑距离（插入、删除、替换各计一次）。"""
    prev = list(range(len(b) + 1))
    for i in range(1, len(a) + 1):
        cur = [i] + [0] * len(b)
        for j in range(1, len(b) + 1):
            cur[j] = min(prev[j] + 1, cur[j - 1] + 1,
                         prev[j - 1] + (a[i - 1] != b[j - 1]))
        prev = cur
    return prev[-1]


def cluster_vote(samples, thresh=1):
    """按编辑距离聚簇后投票，返回代表答案。

    把编辑距离不超过 thresh 的答案看成同一簇（连边），用并查集找出
    全部连通分量，分量权值为成员出现次数之和；取权值最大的分量，
    以其出现次数最多的成员为代表答案。
    """
    counter = Counter(samples)
    words = list(counter)
    parent = list(range(len(words)))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra

    for i in range(len(words)):
        for j in range(i + 1, len(words)):
            if edit_distance(words[i], words[j]) <= thresh:
                union(i, j)

    comp_weight = {}
    for i, w in enumerate(words):
        r = find(i)
        comp_weight[r] = comp_weight.get(r, 0) + counter[w]

    best_r = max(comp_weight, key=comp_weight.get)
    members = [words[i] for i in range(len(words)) if find(i) == best_r]
    return max(members, key=counter.get)


def typo(word, rng):
    """以一次编辑距离生成 word 的变体，六成概率直接返回原词。"""
    if rng.random() < 0.6:
        return word
    i = rng.integers(0, len(word))
    if rng.random() < 0.5:
        return word[:i] + word[i + 1:]
    return word[:i] + ("z" if word[i] != "z" else "q") + word[i + 1:]


WORDS = ["banana", "grape", "peach", "melon", "lemon"]
WRONGS = ["apple", "plum", "cherry", "kiwi", "mango", "fig"]


def simulate_batch(n_questions, K, p_correct, seed):
    """生成一批题目的采样答案，返回三种策略的正确率。

    每条路径以 p_correct 输出正确词的变体，否则输出一个无关错误词。
    """
    rng = np.random.default_rng(seed)
    single = exact = clustered = 0
    for _ in range(n_questions):
        truth = WORDS[rng.integers(0, len(WORDS))]
        samples = []
        for _ in range(K):
            if rng.random() < p_correct:
                samples.append(typo(truth, rng))
            else:
                samples.append(WRONGS[rng.integers(0, len(WRONGS))])
        single += samples[0] == truth
        exact += Counter(samples).most_common(1)[0][0] == truth
        clustered += cluster_vote(samples) == truth
    n = float(n_questions)
    return single / n, exact / n, clustered / n


print("编辑距离示例：banana 与 bananna =", edit_distance("banana", "bananna"),
      "，banana 与 grape =", edit_distance("banana", "grape"))

p_correct = 0.6
K = 9
res = simulate_batch(3000, K, p_correct, 7)
for name, v in zip(["单次采样", "精确多数投票", "编辑距离簇投票"], res):
    print(f"{name}：正确率 {v:.3f}")
print("关键观察：簇投票合并了正确词的拼写变体，正确率高于精确投票。")


投票的收益依赖采样路径的多样性：路径必须各不相同，投票才有意义。一个控制多样性的旋钮叫温度（temperature）。温度越高，模型采样越随机，路径越容易分岔；温度接近 0 时，采样退化成每次都选概率最大的那个词，这种生成方式叫贪婪解码，K 条路径会彼此相同，投票形同虚设。但温度也不是越高越好：分布越平，每条路径单独做对的概率也在下降。下面用一个我们自己写的采样器，把这两个作用都写成温度的函数，观察多数投票准确率随温度的变化。

**温度是怎么让路径分岔的**

模型预测下一个 token 时，会给每个可能的 token 打一个原始分数，这个分数叫 logits，分数越高表示越倾向选它。要把分数变成概率，可以先对每个分数取指数，再除以所有分数的总和，让概率加起来是 1，这个过程叫 softmax。温度 T 就插在 softmax 之前：采样时先让 logits 除以 T，再过 softmax：

$$P(\text{token}_i) = \frac{\exp(z_i / T)}{\sum_j \exp(z_j / T)}$$

T 越大，除出来的数值越接近，分布越平；T 越小，数值差被放大，分布越尖。用一个三分类例子看。假设三个 token 的 logits 是 [2.0, 1.0, 0.0]：

| T | P(token1) | P(token2) | P(token3) | 分布形态 |
|:---|:---|:---|:---|:---|
| 0.5 | 0.867 | 0.117 | 0.016 | 几乎总是 token1 |
| 1.0 | 0.665 | 0.245 | 0.090 | 有偏，但留有余地 |
| 2.0 | 0.506 | 0.307 | 0.186 | 接近均匀，token3 也有机会 |

取 T=1 手算一行。exp 值为 $[e^2, e^1, e^0]$ = [7.39, 2.72, 1]，总和 11.11，除以总和得 [0.665, 0.245, 0.090]。T=0.5 相当于 logits 先乘 2 再 softmax，exp 变成 $[e^4, e^2, e^0]$，差距被放大，token1 的概率接近 0.87。T=2 相当于除以 2，三个概率趋同。

把温度和自洽投票连起来看。T 接近 0 时，每步都选概率最大的 token，五次采样会走出五条几乎相同的路径，投票等于没有投票。T 升高后，概率小的 token 也有机会被选到，路径开始分岔，这正是投票需要的多样性。但温度不是越高越好：分布越平，模型越像随机猜，每条路径单独做对的概率下降。温度的作用是找到"路径足够分岔、又不至于随机"的中间值，模拟里 PaLM 与 UL2 的经验值是 0.5 到 0.7。

In [ ]:
def temperature_accuracy(temp, K, p_good, tau, div_scale, W, n_trials, seed):
    """模拟温度对多数投票的影响。

    temp 升高同时带来路径多样性的上升与单条路径质量的下降，
    两个作用分别用指数形式建模。
    """
    rng = np.random.default_rng(seed)
    p = p_good * np.exp(-temp / tau)          # 单条路径做对概率
    div = 1.0 - np.exp(-temp / div_scale)     # 路径多样性
    hits = 0
    for _ in range(n_trials):
        if rng.random() < div:
            correct = rng.random(K) < p
            wrong = rng.integers(1, W + 1, size=K)
            votes = np.where(correct, 0, wrong)
            counts = np.bincount(votes, minlength=W + 1)
            hits += counts.argmax() == 0
        else:
            hits += rng.random() < p_good     # 低温退化：K 条路径都等于贪婪路径
    return hits / n_trials


temp_grid = np.linspace(0.0, 2.0, 11)
fig, ax = plt.subplots(figsize=(6.4, 3.6))
for K in [1, 5, 40]:
    acc = [temperature_accuracy(t, K, 0.6, 1.5, 0.6, 4, 1500, 11)
           for t in temp_grid]
    ax.plot(temp_grid, acc, marker="o", label=f"K = {K}")
ax.set_xlabel("temperature")
ax.set_ylabel("accuracy")
ax.set_title("Temperature and the number of sampled paths")
ax.legend()
plt.tight_layout()
plt.show()

print("关键观察：路径越多，准确率对温度越不敏感；温度过高时单条路径质量")
print("跌穿 1/W，多数投票也跟着失效。实践里 PaLM 与 UL2 的采样温度取在 0.5 到 0.7。")


多条路径的票数分布本身就是信息。直觉上，多条路径都指向同一个答案，说明模型对这个答案很有把握；路径各说各的，说明模型拿不准。把这种把握量化：先看票最多的答案占了多少票，算出它的占比，占比越接近 1，说明路径越一致。这个占比就叫一致性分数。我们合成一批难易混合的题目，验证一致性分数和"多数投票是否正确"是不是相关，再演示一个简单策略：一致性低于某个阈值的题，宁可放过不答。在 Agent 框架里，这个信号正是模型判断"什么时候不该自己拿主意"的依据。

In [ ]:
def consistency_correctness(n_questions, K, W, seed):
    """模拟难易混合的题目，返回 (最大票占比, 多数是否正确) 数组。"""
    rng = np.random.default_rng(seed)
    fractions, corrects = [], []
    for _ in range(n_questions):
        p = 0.90 if rng.random() < 0.45 else 0.12   # 四成易题，六成难题
        ok = rng.random(K) < p
        wrong = rng.integers(1, W + 1, size=K)
        votes = np.where(ok, 0, wrong)
        counts = np.bincount(votes, minlength=W + 1)
        fractions.append(counts.max() / K)
        corrects.append(counts.argmax() == 0)
    return np.array(fractions), np.array(corrects)


K, W = 20, 6
fractions, corrects = consistency_correctness(4000, K, W, 41)
r = np.corrcoef(fractions, corrects)[0, 1]
acc_all = corrects.mean()
acc_accept = corrects[fractions >= 0.5].mean()

edges = np.linspace(0, 1, 21)
xf, yc = [], []
for lo, hi in zip(edges[:-1], edges[1:]):
    mask = (fractions >= lo) & (fractions < hi)
    if mask.sum() > 10:
        xf.append(fractions[mask].mean())
        yc.append(corrects[mask].mean())

fig, ax = plt.subplots(figsize=(6, 3.6))
ax.plot(xf, yc, marker="o")
ax.set_xlabel("max vote fraction (consistency)")
ax.set_ylabel("probability majority is correct")
ax.set_title("Consistency as an uncertainty signal")
plt.tight_layout()
plt.show()

print(f"一致性分数与正确与否的相关系数：{r:.3f}")
print(f"全部题目正确率 {acc_all:.3f}，只采纳一致性≥0.5 的题正确率 {acc_accept:.3f}")
print("关键观察：票数越集中越可能正确，一致性可以作为拒绝回答的阈值。")


前面几节算的都是算术题。CoT 论文还有一个结果，落在另一种任务上，这种任务不涉及计算，只把符号按规则拼接、变换，叫符号任务。它观察的现象叫长度外推：样例里只给短的输入，测试时输入变长，模型能不能继续做对。任务本身很简单：把名字每个词的末字母取出来拼在一起。输入 Mary Jane Smith，分步得到 y、e、h，输出 yeh。few-shot 样例只用 2 个词的短名字，模型学到的是"逐词处理"这条规则，遇到 3 词、4 词的长名字也能沿用。直接给答案的模型只是记下了短样例，长度一变长就失效。下面用规则模拟分步与直接两种策略，对比长度外推下的正确率。

**末字母任务：一个能算的符号例子**

CoT 论文的长度外推实验用的是末字母拼接任务：输入一个名字，取出每个词的最后一个字母拼起来。Mary Jane Smith 分步得到 y、e、h，输出 yeh。few-shot 样例里只给 2 个词的名字，模型从样例学到"对每个词，输出它的末字母"这条规则。这条规则不依赖词的个数，3 词、4 词的名字也能逐词沿用。直接给答案的模型没有学到这条规则，它只是记住了短样例，遇到更长的名字只能猜。

这个差别可以用两个概率模型算出来。先想直接作答：它要一次处理整串词，词越多，中间任何一处出错的机会越大，所以正确率随长度指数衰减，我们用 $p_{\text{direct}}(L) = 0.7\, e^{-0.8(L-2)}$ 来描述这个衰减。再想分步作答：它每词做对概率是 0.9，需要 L 个词都做对，正确率是 $0.9^L$。代入各长度：

| 长度（词数） | 直接作答 | 分步作答 |
|:---|:---|:---|
| 2 | 0.70 | 0.81 |
| 3 | 0.31 | 0.73 |
| 4 | 0.14 | 0.66 |
| 5 | 0.06 | 0.59 |

长度从 2 变到 5，直接作答的正确率从 0.70 降到 0.06，分步作答只从 0.81 降到 0.59。差别来自任务在两种策略里的位置：直接作答把"处理多少个词"当成任务难度的一部分，词越多越容易出错；分步作答把任务拆成一个个独立的"取末字母"小步，每步的难度不随长度变化，正确率随长度下降只是因为步数变多。

这个例子说明思维链外推的来源：分步策略把"长度外推"转化为"每步都做见过的短任务"。模型不需要见过 5 词的名字，它只需要把 5 次"取末字母"依次做对。

In [ ]:
def direct_accuracy(length, p0, decay, seed):
    """直接作答：正确率随长度指数衰减，模拟对长样本的泛化失败。"""
    rng = np.random.default_rng(seed)
    p = p0 * np.exp(-decay * (length - 2))
    return float((rng.random(3000) < p).mean())


def stepwise_accuracy(length, p_step, seed):
    """分步作答：每步做对概率 p_step，全对才得分。"""
    rng = np.random.default_rng(seed)
    steps = rng.random((3000, length)) < p_step
    return float(steps.all(axis=1).mean())


lengths = [2, 3, 4, 5]
direct = [direct_accuracy(L, 0.7, 0.8, 21) for L in lengths]
stepwise = [stepwise_accuracy(L, 0.9, 22) for L in lengths]

fig, ax = plt.subplots(figsize=(6, 3.6))
ax.plot(lengths, direct, marker="o", label="direct answer")
ax.plot(lengths, stepwise, marker="s", label="step-by-step")
ax.set_xlabel("number of words (out of distribution)")
ax.set_ylabel("accuracy")
ax.set_title("Length extrapolation on symbol tasks")
ax.set_xticks(lengths)
ax.legend()
plt.tight_layout()
plt.show()

print("直接作答在长度", lengths, "上的正确率：", [f"{v:.2f}" for v in direct])
print("分步作答在长度", lengths, "上的正确率：", [f"{v:.2f}" for v in stepwise])
print("关键观察：直接作答随长度外推崩溃，分步作答把长样本拆成短步骤后保持稳健。")


前面的实验反复出现同一个现象：小模型用思维链没有收益甚至倒退，模型规模跨过某个门槛后，收益突然出现。这种"过了门槛才突然变强"的现象叫涌现：能力在一个规模段贴近随机，跨过某个临界规模后，跳到明显高于随机。涌现论文（Wei 等人，2022）把这类能力收集成清单，思维链与自洽解码都在其中。

这里有一个值得追问的地方：看到的跳变，可能只是度量方式造成的。度量就是给模型回答打分的方式，打分的方式不同，看到的曲线形状就不同。一种只分对错的度量叫二值度量：整段答案和标准答案完全一致才算对，差一点都不算，精确匹配（exact match）就是典型的例子。另一种度量给部分分，比如逐 token 精度：一个 token 一个 token 地数，数对的占多少，部分正确也算分。Yi 等人（2204.07646）用逐 token 精度这类度量重画同一批曲线，发现曲线变得平滑可预测，据此认为"涌现"是二值度量制造的假象。两边对"底层能力是不是平滑增长"没有分歧，分歧在于任务级准确率该用什么度量来读。下面在合成数据上把这场争论重演一遍。

构造一条随规模平滑增长的底层能力：把答案写成一个 5 token 的序列，每个 token 独立做对，单 token 正确率从 0.2 平滑涨到 0.9。用两种度量读它：逐 token 精度直接等于单 token 正确率，是一条平滑的直线；精确匹配要求 5 个 token 全对，正确率是单 token 正确率的 5 次方，在单 token 正确率约 0.5 的位置出现明显跳变。同一个底层能力，两种读数。


**同一个能力，两种读法：手算五个规模点**

把这条 toy 能力在五个规模点上分别用两种度量算一遍。单 token 正确率 p 随规模线性增长，取 p = 0.41、0.48、0.55、0.62、0.69 五档，每档只涨 0.07：

| 单 token 正确率 p | 逐 token 精度 = p | 精确匹配 = $p^5$ |
|:---|:---|:---|
| 0.41 | 0.410 | 0.012 |
| 0.48 | 0.480 | 0.025 |
| 0.55 | 0.550 | 0.050 |
| 0.62 | 0.620 | 0.092 |
| 0.69 | 0.690 | 0.156 |

逐 token 精度就是 p 本身，五个点排成一条直线，均匀上升。精确匹配是 p 的 5 次方，五个点从 0.012 涨到 0.156，越往右涨得越陡：p 从 0.48 到 0.55 只涨了 14.6%，精确匹配从 0.025 涨到 0.050，几乎翻倍；p 从 0.55 到 0.62 只涨 12.7%，精确匹配又涨了 82%。

底层能力每步只涨了 0.07，没有任何突变；看起来的跳变来自 p 的 5 次方这个非线性度量把小的增量放大了。更准确地说，精确匹配是二值度量：5 个 token 全对才算对，错一个就整句记零分，部分得分被全部抹掉；逐 token 精度保留部分得分。同一个底层能力，度量不同，读出来的形状就不同。这就是"涌现是度量造成的假象"这句话的完整含义——底层平滑，度量把平滑放大成了陡峭。

In [ ]:
s = np.linspace(0, 1, 200)
p_tok = 0.2 + 0.7 * s            # 单 token 正确率：平滑线性增长
L = 5
exact_match = p_tok ** L          # 精确匹配：全对才得分
per_token = p_tok                 # 逐 token：部分对也给分

fig, ax = plt.subplots(figsize=(6.4, 3.6))
ax.plot(s, per_token, label="per-token accuracy")
ax.plot(s, exact_match, label="exact match (5 tokens)")
ax.set_xlabel("scale")
ax.set_ylabel("accuracy")
ax.set_title("The same ability under two metrics")
ax.legend()
plt.tight_layout()
plt.show()

print("关键观察：exact match 曲线呈现相变形态，per-token 曲线平滑；")
print("底层能力没有任何突变，跳变来自度量把部分得分全部抹掉了。")


同样的机理在多步任务上更明显。假设每步做对概率 p 平滑增长，最终正确率是 p 的 L 次方。L 越大，低 p 段被压得越平，跳变越陡。这也是思维链只在足够大的模型上生效的原因之一：小模型的多步正确率被压得接近零，跨过阈值后才开始显现。画出 L = 1、3、5 三条曲线。


**步骤越多，放大越狠**

同样的复合过程在更长的任务上更极端。设每步做对概率为 p，最终正确率是 $p^L$。固定几档 p，看不同 L 下的最终正确率：

| p | L=1（一次作答） | L=3 | L=5 |
|:---|:---|:---|:---|
| 0.5 | 0.500 | 0.125 | 0.031 |
| 0.6 | 0.600 | 0.216 | 0.078 |
| 0.7 | 0.700 | 0.343 | 0.168 |
| 0.8 | 0.800 | 0.512 | 0.328 |

看 L=5 那一列。p 从 0.5 到 0.6 只涨 0.1，最终正确率从 0.031 涨到 0.078，变成原来的 2.5 倍；p 从 0.7 到 0.8 又涨 0.1，最终正确率从 0.168 涨到 0.328，接近翻倍。同样的 0.1 增量，在 p 大的一侧被放大得更明显。

这解释了两个观察。其一，步骤越多，低 p 段的最终正确率被压得越接近零，曲线越像"要么贴近零、要么起来"的跳变，这就是 L 大时涌现看起来更陡的原因。其二，思维链只在足够大的模型上生效：小模型每步做对概率低，L 步连乘后正确率被压到接近随机水平，看不出效果；模型变大后每步概率抬高，连乘的结果才从贴近零转为明显高于随机。

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 3.6))
for L in [1, 3, 5]:
    ax.plot(s, p_tok ** L, label=f"{L} steps: p^L")
ax.set_xlabel("per-step accuracy (smooth)")
ax.set_ylabel("task accuracy")
ax.set_title("Compounding single-step ability into task accuracy")
ax.legend()
plt.tight_layout()
plt.show()

print("关键观察：同样的平滑单步能力，步骤越多，最终准确率曲线越接近跳变。")


这场争论留下可操作的经验：底层能力按平滑曲线增长，而用户关心的任务级准确率由复合过程决定；看起来的跳变，是复合过程被二值度量放大后的结果。涌现论文用另一类证据补上反方没说到的部分。训练时有一个量衡量模型预测和真实答案的差距，叫损失，交叉熵是常见的一种。用交叉熵看，小规模段的损失确实在稳步下降，说明能力在积累；而且分类任务用精确匹配度量，也会出现涌现。这两点说明，度量并不能完全解释跳变。更稳妥的读法是两边各占一半：能力平滑积累，度量决定它如何被看见。

前几节做的都是在提示层面激发推理：用户构造样例、调温度、做投票，能力上限受提示约束，换一道新题常常要重新设计提示。2024 年起的推理模型（OpenAI o1、DeepSeek-R1）走的是另一条路：把"多步推理"搬进训练阶段，让模型自己学会在内部思考。训练用的方法叫强化学习：给模型的回答打分，回答得好就鼓励，回答得差就惩罚，模型在反复练习中学会把思维链写得更长更准。这样训练完之后，多步推理已经内化成权重的一部分，推理时只需一句简短指令，思维链不再由提示提供，而是由模型自己生成。

两代思路的差异可以并排看：

| 维度 | 提示激发时代 | 推理模型时代 |
|:---|:---|:---|
| 推理从哪来 | few-shot 样例里的中间步骤 | 训练阶段的强化学习 |
| 要改权重吗 | 不改，纯提示与解码侧改动 | 改，后训练阶段学习 |
| 路径怎么来 | 用户写样例示范 | 模型自己生成思维链 |
| 推理预算 | 采样条数、温度 | 隐藏的思考 token 数量 |
| 不确定性 | 自洽投票的票数占比 | 模型自报置信度 |

两个时代并不对立。提示技巧在推理模型上依然有效，自洽投票作为“知道自己不知道”的信号仍是 Agent 的重要组件：取得一批采样，先看票数是否集中，集中才采纳，不集中宁可拒绝回答。下一讲进入数学推理（AlphaProof），把这条路推到证明级别。


## 小结

这一讲沿着"推理能力如何被调用"走了一遍：

- [ ] 推理能力来自预训练，提示格式决定它能否被调用，标准提示只是能力下界
- [ ] 把样例改成 ⟨问题, 推理步骤, 答案⟩，思维链就能激发多步推理，且无需训练
- [ ] 消融说明关键变量是"用自然语言按顺序写出步骤"，方程、省略号、后置推理都没有增益
- [ ] 多次采样加多数投票把多条路径在答案处聚合，错误路径很少汇合
- [ ] 编辑距离聚类投票能合并同一答案的不同写法，比精确投票更稳
- [ ] 温度平衡路径多样性与质量，路径越多对温度越不敏感
- [ ] 一致性分数与正确与否正相关，可作为"知道不知道"的置信度信号
- [ ] 符号任务的逐词规则让思维链在更长序列上外推，直接作答则崩溃
- [ ] 同一底层能力在不同度量下呈现涌现或平滑，度量会影响结论


## 作业

> 可以让 AI 帮忙解释思路，但不建议直接让 AI "做完这道题"。

三道题分别对应思维链的解析与消融、自洽聚合、涌现度量。每道题有一段带空位的代码，补全后运行 assert 即可自检。


In [ ]:
import re

# 作业 1：补全答案解析正则与多步正确率公式
# 空位 1：匹配"最终答案是 N"这类收尾，应补 r"最终答案是\s*(\d+)"
PATTERN = r"最终答案是\s*(\d+)"

def parse_ans(text):
    m = re.search(PATTERN, text)
    return int(m.group(1)) if m else None

assert parse_ans("答案是 5。分步算得 5。最终答案是 11。") == 11
assert parse_ans("最终答案是 19。") == 19

# 空位 2：L 步全对才得分，应补 p_step ** L
p_step, L = 0.8, 3
p_cot = p_step ** L
assert p_cot > 0.5, "分步后正确率应明显高于一次作答 0.25"
print(f"解析与公式正确：3 步全对概率 {p_cot:.3f}，高于直接作答 0.25。")
# 小提示：正则只捕获"最终答案是"后面的整数；多步正确率是每步概率连乘。


In [ ]:
# 作业 2：补全多数投票与编辑距离
# 空位 1：取出现次数最多的答案，应补 counts.argmax()
def majority(answers, W=8):
    votes = np.array(answers)
    counts = np.bincount(votes, minlength=W + 1)
    return int(counts.argmax())

assert majority([3, 3, 5, 3, 5]) == 3
assert majority([1, 2, 2]) == 2

# 空位 2：替换代价只在字符不同时计 1，应补 a[i - 1] != b[j - 1]
def edit_distance2(a, b):
    prev = list(range(len(b) + 1))
    for i in range(1, len(a) + 1):
        cur = [i] + [0] * len(b)
        for j in range(1, len(b) + 1):
            cur[j] = min(prev[j] + 1, cur[j - 1] + 1,
                         prev[j - 1] + (a[i - 1] != b[j - 1]))
        prev = cur
    return prev[-1]

assert edit_distance2("banana", "bananna") == 1
assert edit_distance2("banana", "grape") > 1
print("多数投票与编辑距离实现正确，同一答案的不同写法在簇内合并。")
# 小提示：多数投票数的是出现次数；编辑距离的替换分支只在字符不同时加一。


In [ ]:
# 作业 3：补全两种度量并实现涌现检测器
# 空位 1：精确匹配要求 L 个 token 全对，应补 p ** L
# 空位 2：逐 token 精度对部分对也得分，应补 p
def metrics(p, L):
    exact = p ** L
    per_token = p
    return exact, per_token

# 涌现检测器：后半段增速相对前半段的倍数超过阈值即判涌现
def detect_emergent(x, y, thr=2.0):
    half = len(x) // 2
    slope_front = (y[half] - y[0]) / (x[half] - x[0])
    slope_back = (y[-1] - y[half]) / (x[-1] - x[half])
    return slope_back / (slope_front + 1e-9) > thr

s = np.linspace(0, 1, 200)
p_tok = 0.2 + 0.7 * s
exact, per_tok = metrics(p_tok, 5)
assert detect_emergent(s, exact), "exact match 曲线应判为涌现"
assert not detect_emergent(s, per_tok), "per-token 曲线应判为平滑"
print("度量检查通过：同一个底层能力，exact match 呈现涌现，per-token 呈现平滑。")
# 小提示：前后两段各算一次平均增速；per-token 是直线，两段增速相同。


## 参考资料

- [Chain-of-Thought Prompting Elicits Reasoning in Large Language Models](https://arxiv.org/abs/2201.11903)（Wei 等人，2022）— 本讲锚点：few-shot 思维链提示、消融与规模涌现
- [Self-Consistency Improves Chain of Thought Reasoning](https://arxiv.org/abs/2203.11171)（Wang 等人，2022）— sample-and-marginalize：采样多条路径再多数投票
- [Emergent Abilities of Large Language Models](https://arxiv.org/abs/2206.07682)（Wei 等人，2022）— 涌现能力的定义、证据汇总与自我批判
- [Are Emergent Abilities of Large Language Models a Mirage?](https://arxiv.org/abs/2204.07646)（Yi 等人，2022）— 反方观点：不连续度量制造的涌现假象，与本讲第三节对照
- [Training Verifiers to Solve Math Word Problems](https://arxiv.org/abs/2110.14168)（Cobbe 等人，2021）— GSM8K 数据集与"微调加验证器"基线
- [Large Language Models are Zero-Shot Reasoners](https://arxiv.org/abs/2205.11916)（Kojima 等人，2022）— 零样本思维链：Let's think step by step
- [Least-to-Most Prompting Enables Complex Reasoning](https://arxiv.org/abs/2205.10625)（Zhou 等人，2022）— 思维链的进阶分解策略
- [Show Your Work: Scratchpads for Intermediate Computation](https://arxiv.org/abs/2112.00114)（Nye 等人，2021）— 中间计算预测，在极小模型上就涌现
- [Finetuned Language Models are Zero-Shot Learners](https://arxiv.org/abs/2109.01652)（Wei 等人，2021）— FLAN 指令微调的涌现
- [BIG-bench: Beyond the Imitation Game](https://arxiv.org/abs/2206.04615)（Srivastava 等人，2022）— 涌现证据的主要来源
